# Solutions · 14 · Crystallization

Worked solutions to the exercises of [notebook 14](../notebooks/14_crystallization.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engsep import crystallization as cr
from engsep import datasets
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Sodium sulphate has a solubility maximum near 32 degC (the decahydrate-anhydrous transition). Using the table, find the temperature range over which *cooling* a saturated solution gives crystals and the range over which *heating* does. Which salt is best crystallised by evaporation at 80 degC?

In [2]:
T = np.linspace(0, 100, 201)
s = cr.solubility("Na2SO4", T)
i_max = int(np.argmax(s))
print(f"Na2SO4 solubility peaks at about {T[i_max]:.0f} degC in this coarse table ({s[i_max]:.1f} g/100 g; the true maximum is at 32.4 degC): below the peak, cooling crystallises; above it, heating does (retrograde solubility)")
for salt in cr.SOLUBILITY:
    print(f"{salt:<8}: solubility at 80 degC {cr.solubility(salt, 80.0):6.1f} g/100 g water -> crystals per kg water evaporated at 80 degC: {cr.solubility(salt, 80.0)/100*1000:.0f} g")

Na2SO4 solubility peaks at about 40 degC in this coarse table (48.8 g/100 g; the true maximum is at 32.4 degC): below the peak, cooling crystallises; above it, heating does (retrograde solubility)
KNO3    : solubility at 80 degC  169.0 g/100 g water -> crystals per kg water evaporated at 80 degC: 1690 g
NaCl    : solubility at 80 degC   38.4 g/100 g water -> crystals per kg water evaporated at 80 degC: 384 g
KCl     : solubility at 80 degC   51.1 g/100 g water -> crystals per kg water evaporated at 80 degC: 511 g
CuSO4   : solubility at 80 degC   55.0 g/100 g water -> crystals per kg water evaporated at 80 degC: 550 g
Na2SO4  : solubility at 80 degC   43.7 g/100 g water -> crystals per kg water evaporated at 80 degC: 437 g
sucrose : solubility at 80 degC  362.1 g/100 g water -> crystals per kg water evaporated at 80 degC: 3621 g
K2Cr2O7 : solubility at 80 degC   73.0 g/100 g water -> crystals per kg water evaporated at 80 degC: 730 g
NH4Cl   : solubility at 80 degC   65.6 g/100 g water

Sodium sulphate's solubility rises to the decahydrate-anhydrous transition (32.4 degC; the 20-degree table
puts the peak near 40) and then falls slightly: above the transition, heating crystallises it - the retrograde solubility used in some salt-recovery
processes. For evaporative crystallisation the most soluble salt at the evaporation temperature gives the
most crystals per kilogram of water removed: sucrose and potassium nitrate far ahead of sodium chloride.

## Exercise 2

The MSMPR data are re-sieved with a coarser set (drop every other size). Refit G and n0 and estimate the uncertainty of the dominant size; then compute how much the residence time must change to shift the dominant size by 20 %.

In [3]:
d = pd.read_csv(datasets.path("csd_msmpr.csv"), comment="#")
full = cr.msmpr_from_csd(d.L_m, d.n_per_m3_m, 2400.0)
coarse = cr.msmpr_from_csd(d.L_m[::2], d.n_per_m3_m[::2], 2400.0)
print(f"all sizes: G = {full['G']*1e9:.2f} nm/s, L_D = {cr.msmpr_dominant_size(full['G'], 2400.0)*1e6:.0f} um; every other size: G = {coarse['G']*1e9:.2f} nm/s, L_D = {cr.msmpr_dominant_size(coarse['G'], 2400.0)*1e6:.0f} um")
print(f"a 20 % larger dominant size needs tau = {1.2 * 2400 / 60:.0f} min instead of 40 (at the same growth rate)")

all sizes: G = 8.00 nm/s, L_D = 58 um; every other size: G = 7.99 nm/s, L_D = 58 um
a 20 % larger dominant size needs tau = 48 min instead of 40 (at the same growth rate)


Dropping half the sieve fractions changes the growth rate by a few percent - the slope of a straight line
on a semi-log plot is robust - and the dominant size with it. Residence time scales the dominant size
linearly at constant growth rate, so 20 % more size means 20 % more time in the crystalliser.

## Exercise 3

**Implement it yourself:** compute the third moment of the measured population density by trapezoidal integration and compare the magma density it implies (k_v = pi/6, rho_c = 2660 kg/m3) with the closed-form 6 n0 (G tau)^4.

In [4]:
L, n = d.L_m.to_numpy(), d.n_per_m3_m.to_numpy()
trap = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
m3 = trap(n * L**3, L)
k_v, rho_c = np.pi / 6, 2660.0
fit = cr.msmpr_from_csd(L, n, 2400.0)
m3_closed = 6 * fit["n0"] * (fit["G"] * 2400.0) ** 4
print(f"third moment from the data: {m3:.3e} m3/m3 (magma density {k_v*rho_c*m3:.0f} kg/m3); closed form 6 n0 (G tau)^4: {m3_closed:.3e} - the data stop at 800 um and miss the tail")

third moment from the data: 1.566e-01 m3/m3 (magma density 218 kg/m3); closed form 6 n0 (G tau)^4: 1.649e-01 - the data stop at 800 um and miss the tail


The measured moments fall short of the closed form because the sieves stop at 800 micrometres and the mass
distribution has a long tail; integrating data means integrating only what was measured. The closed form
extrapolates the exponential - one more reason to fit the line rather than sum the fractions.